# 02 — 5-Core 过滤、序列构建与数据划分

本 Notebook 对 8 个 Amazon 领域 + Goodreads 执行以下操作：

1. **迭代式 5-core 过滤**：反复移除交互次数 < 5 的用户和物品，直到收敛
2. **物品 ID 重编号**：从 1 开始的连续整数（0 保留作为 padding）
3. **按时间排序**：构建每个用户的交互序列，最大长度限制为 10
4. **Leave-one-out 划分**：最后 1 个 → test，倒数第 2 个 → val，其余 → train
5. **生成评估文件**：`data.txt`, `train_data.txt`, `val_data.txt`, `test_data.txt`, `item_titles.json`
6. **验证**：与论文 Table 1/2 的统计数据对照

## 论文中的统计数据 (5-core 过滤后)

| 数据集 | 物品数 | 交互数 |
|--------|--------|--------|
| Games (Video_Games) | 9,517 | 153,221 |
| Arts (Arts_Crafts_and_Sewing) | 12,454 | 132,566 |
| Movies (Movies_and_TV) | 13,190 | 136,471 |
| Home (Home_and_Kitchen) | 33,478 | 256,001 |
| Electronics | 20,150 | 197,984 |
| Tools (Tools_and_Home_Improvement) | 19,964 | 159,969 |
| Goodreads | 4,550 | 158,347 |

## 输出文件
```
data/{Category}/5-core/downstream/
├── data.txt           # 全量用户序列
├── train_data.txt     # 训练序列 (去掉最后2个)
├── val_data.txt       # 验证序列 (去掉最后1个)
├── test_data.txt      # 完整序列
└── item_titles.json   # {"1": "title1", "2": "title2", ...}

data/Goodreads/clean/
└── (同上5个文件)
```

## 0. 环境准备

In [ ]:
import os
import json
import pandas as pd
import numpy as np
from pathlib import Path
from tqdm import tqdm
from collections import defaultdict, Counter

# 项目根目录
PROJECT_ROOT = Path(os.path.abspath(os.path.join(os.getcwd(), '..')))
DATA_DIR = PROJECT_ROOT / 'data'
INTERMEDIATE_DIR = DATA_DIR / 'intermediate'

print(f"项目根目录: {PROJECT_ROOT}")
print(f"中间文件目录: {INTERMEDIATE_DIR}")

# 论文中的统计数据 (用于验证)
PAPER_STATS = {
    'Video_Games': {'items': 9517, 'interactions': 153221},
    'Arts_Crafts_and_Sewing': {'items': 12454, 'interactions': 132566},
    'Movies_and_TV': {'items': 13190, 'interactions': 136471},
    'Home_and_Kitchen': {'items': 33478, 'interactions': 256001},
    'Electronics': {'items': 20150, 'interactions': 197984},
    'Tools_and_Home_Improvement': {'items': 19964, 'interactions': 159969},
    'Sports_and_Outdoors': {},  # 论文未报告
    'Baby_Products': {},        # 论文未报告
    'Goodreads': {'items': 4550, 'interactions': 158347},
}

## 1. 5-Core 过滤算法

标准 5-core 过滤：
- 反复移除交互次数 < 5 的用户
- 反复移除交互次数 < 5 的物品
- 循环直到没有记录被移除

这是推荐系统研究中的标准做法，确保每个用户和物品都有足够的交互数据。

In [ ]:
def five_core_filter(df: pd.DataFrame, user_col: str = 'user_id', item_col: str = 'parent_asin', min_count: int = 5) -> pd.DataFrame:
    """
    迭代式 5-core 过滤。
    反复移除交互次数 < min_count 的用户和物品，直到收敛。
    
    Args:
        df: 交互 DataFrame，需包含 user_col 和 item_col 列
        user_col: 用户 ID 列名
        item_col: 物品 ID 列名
        min_count: 最小交互次数阈值
    Returns:
        过滤后的 DataFrame
    """
    print(f"  📏 初始: {len(df):,} 条交互, {df[user_col].nunique():,} 用户, {df[item_col].nunique():,} 物品")
    
    iteration = 0
    while True:
        iteration += 1
        prev_len = len(df)
        
        # 过滤交互次数 < min_count 的用户
        user_counts = df[user_col].value_counts()
        valid_users = user_counts[user_counts >= min_count].index
        df = df[df[user_col].isin(valid_users)]
        
        # 过滤交互次数 < min_count 的物品
        item_counts = df[item_col].value_counts()
        valid_items = item_counts[item_counts >= min_count].index
        df = df[df[item_col].isin(valid_items)]
        
        removed = prev_len - len(df)
        print(f"    第 {iteration} 轮: 移除 {removed:,} 条 → 剩余 {len(df):,} 条, {df[user_col].nunique():,} 用户, {df[item_col].nunique():,} 物品")
        
        if removed == 0:
            break
    
    print(f"  ✅ 5-core 过滤完成 (共 {iteration} 轮): {len(df):,} 条交互, {df[user_col].nunique():,} 用户, {df[item_col].nunique():,} 物品")
    return df.reset_index(drop=True)

## 2. 序列构建与数据划分

关键步骤：
1. 按用户分组，每组按时间排序
2. 物品 ID 重编号（ASIN → 从 1 开始的整数）
3. 截断序列至最大长度（论文中 max_seq_length = 10）
4. Leave-one-out 划分：
   - `data.txt`：完整序列（最多 11 个 item，因为 max_seq_length+1=11）
   - `train_data.txt`：去掉最后 2 个（即序列 [:-2]，最后加上倒数第3个作为标签 → 实际为 [:-2]+[倒数第2个] 不对...）

让我们仔细看看 `seqrec/recdata.py` 中的数据使用方式：
```python
# 在 SequenceDataset.__getitem__ 中:
item_seq = seq[:-1]   # 前面的是输入序列
labels = seq[-1]      # 最后一个是标签
```

所以每个文件中的每行是一个完整序列（包含输入+标签），最后一个元素是要预测的标签。

**Leave-one-out 划分逻辑**：
- 假设用户完整交互序列为: `[a, b, c, d, e]`
- `data.txt`: `a b c d e` (完整)
- `test_data.txt`: `a b c d e` (序列 + 最后一个作为标签)
- `val_data.txt`: `a b c d` (序列 + 倒数第2个作为标签)
- `train_data.txt`: `a b c` (序列 + 倒数第3个作为标签)

In [ ]:
MAX_SEQ_LENGTH = 10  # 论文中最大序列长度

def build_sequences_and_split(
    df: pd.DataFrame,
    item_titles_map: dict,
    user_col: str = 'user_id',
    item_col: str = 'parent_asin',
    time_col: str = 'timestamp',
    max_seq_length: int = MAX_SEQ_LENGTH
):
    """
    构建用户交互序列并执行 leave-one-out 划分。
    
    Args:
        df: 5-core 过滤后的交互 DataFrame
        item_titles_map: 原始物品ID → 标题 的映射
        user_col, item_col, time_col: 列名
        max_seq_length: 最大序列长度
    
    Returns:
        dict with keys: 'data', 'train', 'val', 'test', 'item_titles', 'new_id_to_title', 'stats'
    """
    # Step 1: 物品 ID 重编号 (从 1 开始)
    unique_items = sorted(df[item_col].unique())
    item_to_new_id = {item: idx + 1 for idx, item in enumerate(unique_items)}
    num_items = len(unique_items)
    
    # 构建 new_id → title 映射
    new_id_to_title = {}
    missing_titles = 0
    for orig_id, new_id in item_to_new_id.items():
        title = item_titles_map.get(orig_id, None)
        if title:
            new_id_to_title[str(new_id)] = title
        else:
            new_id_to_title[str(new_id)] = f"Unknown_Item_{orig_id}"
            missing_titles += 1
    
    if missing_titles > 0:
        print(f"  ⚠️  {missing_titles} 个物品缺少标题，已用占位符替代")
    
    # Step 2: 按用户分组，按时间排序
    df = df.sort_values([user_col, time_col])
    
    data_seqs = []       # 完整序列 (data.txt)
    train_seqs = []      # 训练序列
    val_seqs = []        # 验证序列
    test_seqs = []       # 测试序列
    
    total_interactions = 0
    
    for user, group in tqdm(df.groupby(user_col), desc="构建序列"):
        # 获取该用户的物品序列（按时间排序）
        items = group[item_col].tolist()
        
        # 转换为新 ID
        item_ids = [item_to_new_id[item] for item in items]
        
        # 截断至最大长度 + 1 (max_seq_length 个历史 + 1 个标签)
        item_ids = item_ids[-(max_seq_length + 1):]
        
        # 需要至少 3 个交互才能有 train/val/test
        if len(item_ids) < 3:
            continue
        
        total_interactions += len(item_ids)
        
        # data.txt: 完整序列
        data_seqs.append(item_ids)
        
        # test: 完整序列 (最后一个是测试标签)
        test_seqs.append(item_ids)
        
        # val: 去掉最后一个 (倒数第2个成为验证标签)
        val_seqs.append(item_ids[:-1])
        
        # train: 去掉最后两个 (倒数第3个成为训练标签)
        train_seqs.append(item_ids[:-2])
    
    stats = {
        'num_users': len(data_seqs),
        'num_items': num_items,
        'num_interactions': total_interactions,
        'avg_seq_length': np.mean([len(s) for s in data_seqs]),
    }
    
    print(f"  📊 序列构建完成: {stats['num_users']:,} 用户, {stats['num_items']:,} 物品, {stats['num_interactions']:,} 交互")
    print(f"  📊 平均序列长度: {stats['avg_seq_length']:.2f}")
    
    return {
        'data': data_seqs,
        'train': train_seqs,
        'val': val_seqs,
        'test': test_seqs,
        'item_titles': new_id_to_title,
        'item_to_new_id': item_to_new_id,
        'stats': stats,
    }

## 3. 文件写入工具

将序列数据和物品标题写入最终格式。

In [ ]:
def save_sequences_to_file(sequences: list, filepath: Path):
    """将序列写入文本文件，每行一个用户序列，空格分隔的整数 ID。"""
    filepath.parent.mkdir(parents=True, exist_ok=True)
    with open(filepath, 'w', encoding='utf-8') as f:
        for seq in sequences:
            f.write(' '.join(map(str, seq)) + '\n')
    print(f"    ✅ 已写入: {filepath.name} ({len(sequences):,} 行)")


def save_item_titles_json(item_titles: dict, filepath: Path):
    """
    保存物品标题 JSON 文件。
    格式: {"1": "title1", "2": "title2", ...}
    key 从 "1" 开始，不含 "0"。
    """
    filepath.parent.mkdir(parents=True, exist_ok=True)
    
    # 验证: 不应包含 "0"
    assert "0" not in item_titles, "item_titles 不应包含 ID '0'（0 保留作为 padding）"
    
    with open(filepath, 'w', encoding='utf-8') as f:
        json.dump(item_titles, f, ensure_ascii=False, indent=2)
    print(f"    ✅ 已写入: {filepath.name} ({len(item_titles):,} 个物品)")


def save_dataset_files(result: dict, output_dir: Path):
    """保存一个数据集的所有评估文件。"""
    output_dir.mkdir(parents=True, exist_ok=True)
    
    save_sequences_to_file(result['data'], output_dir / 'data.txt')
    save_sequences_to_file(result['train'], output_dir / 'train_data.txt')
    save_sequences_to_file(result['val'], output_dir / 'val_data.txt')
    save_sequences_to_file(result['test'], output_dir / 'test_data.txt')
    save_item_titles_json(result['item_titles'], output_dir / 'item_titles.json')

## 4. 处理 8 个 Amazon 领域

对每个领域执行完整流程：加载 → 5-core 过滤 → 序列构建 → 划分 → 保存。

In [ ]:
# 定义每个 Amazon 领域的输出路径映射
AMAZON_OUTPUT_PATHS = {
    'Video_Games': 'Video_Games/5-core/downstream',
    'Arts_Crafts_and_Sewing': 'Arts_Crafts_and_Sewing/5-core/downstream',
    'Movies_and_TV': 'Movies_and_TV/5-core/downstream',
    'Home_and_Kitchen': 'Home_and_Kitchen/5-core/downstream',
    'Electronics': 'Electronics/5-core/downstream',
    'Tools_and_Home_Improvement': 'Tools_and_Home_Improvement/5-core/downstream',
    'Sports_and_Outdoors': 'Sports_and_Outdoors/5-core/downstream',
    'Baby_Products': 'Baby_Products/5-core/downstream',
}

# 存储所有处理结果 (后续 Notebook 3 需要用到)
all_results = {}

In [ ]:
for category, output_path in AMAZON_OUTPUT_PATHS.items():
    print(f"\n{'='*70}")
    print(f"🔄 处理 Amazon 领域: {category}")
    print(f"{'='*70}")
    
    # 加载中间文件
    print(f"\n  📂 加载中间文件...")
    df_interactions = pd.read_parquet(INTERMEDIATE_DIR / f"{category}_interactions.parquet")
    df_items = pd.read_parquet(INTERMEDIATE_DIR / f"{category}_items.parquet")
    
    # 构建物品标题映射: parent_asin -> title
    item_titles_map = dict(zip(df_items['parent_asin'], df_items['title']))
    print(f"  📊 原始数据: {len(df_interactions):,} 交互, {df_interactions['user_id'].nunique():,} 用户, {df_interactions['parent_asin'].nunique():,} 物品")
    
    # 只保留有标题的物品的交互
    df_interactions = df_interactions[df_interactions['parent_asin'].isin(item_titles_map.keys())]
    print(f"  📊 保留有标题的: {len(df_interactions):,} 交互")
    
    # 去重: 同一用户对同一物品只保留最早的交互
    df_interactions = df_interactions.sort_values('timestamp').drop_duplicates(
        subset=['user_id', 'parent_asin'], keep='first'
    )
    print(f"  📊 去重后: {len(df_interactions):,} 交互")
    
    # 5-core 过滤
    print(f"\n  🔍 开始 5-core 过滤...")
    df_filtered = five_core_filter(df_interactions, user_col='user_id', item_col='parent_asin')
    
    # 构建序列并划分
    print(f"\n  🔧 构建序列并划分...")
    result = build_sequences_and_split(
        df_filtered, item_titles_map,
        user_col='user_id', item_col='parent_asin', time_col='timestamp'
    )
    
    # 保存文件
    print(f"\n  💾 保存评估文件到 data/{output_path}/")
    save_dataset_files(result, DATA_DIR / output_path)
    
    # 与论文统计对照
    if category in PAPER_STATS and PAPER_STATS[category]:
        paper = PAPER_STATS[category]
        ours = result['stats']
        items_match = '✅' if abs(ours['num_items'] - paper['items']) / paper['items'] < 0.05 else '⚠️'
        inter_match = '✅' if abs(ours['num_interactions'] - paper['interactions']) / paper['interactions'] < 0.05 else '⚠️'
        print(f"\n  📋 与论文对照:")
        print(f"    物品数: 我们={ours['num_items']:,} vs 论文={paper['items']:,} {items_match}")
        print(f"    交互数: 我们={ours['num_interactions']:,} vs 论文={paper['interactions']:,} {inter_match}")
    
    # 保存结果供后续使用
    all_results[category] = result
    
    print(f"\n  ✅ {category} 处理完成!")

## 5. 处理 Goodreads 数据集

Goodreads 作为 OOD 评估数据集，处理流程类似但列名不同。

In [ ]:
print(f"\n{'='*70}")
print(f"🔄 处理 Goodreads 数据集")
print(f"{'='*70}")

# 加载中间文件
print(f"\n  📂 加载中间文件...")
df_gr_interactions = pd.read_parquet(INTERMEDIATE_DIR / "Goodreads_interactions.parquet")
df_gr_items = pd.read_parquet(INTERMEDIATE_DIR / "Goodreads_items.parquet")

# 构建标题映射: book_id -> title
gr_titles_map = dict(zip(df_gr_items['book_id'], df_gr_items['title']))
print(f"  📊 原始数据: {len(df_gr_interactions):,} 交互, {df_gr_interactions['user_id'].nunique():,} 用户, {df_gr_interactions['book_id'].nunique():,} 书籍")

# 去重
df_gr_interactions = df_gr_interactions.sort_values('timestamp').drop_duplicates(
    subset=['user_id', 'book_id'], keep='first'
)
print(f"  📊 去重后: {len(df_gr_interactions):,} 交互")

# 5-core 过滤
print(f"\n  🔍 开始 5-core 过滤...")
df_gr_filtered = five_core_filter(df_gr_interactions, user_col='user_id', item_col='book_id')

# 构建序列并划分
print(f"\n  🔧 构建序列并划分...")
gr_result = build_sequences_and_split(
    df_gr_filtered, gr_titles_map,
    user_col='user_id', item_col='book_id', time_col='timestamp'
)

# 保存文件
print(f"\n  💾 保存评估文件到 data/Goodreads/clean/")
save_dataset_files(gr_result, DATA_DIR / 'Goodreads' / 'clean')

# 与论文对照
paper = PAPER_STATS['Goodreads']
ours = gr_result['stats']
if paper:
    items_match = '✅' if abs(ours['num_items'] - paper['items']) / paper['items'] < 0.05 else '⚠️'
    inter_match = '✅' if abs(ours['num_interactions'] - paper['interactions']) / paper['interactions'] < 0.05 else '⚠️'
    print(f"\n  📋 与论文对照:")
    print(f"    物品数: 我们={ours['num_items']:,} vs 论文={paper['items']:,} {items_match}")
    print(f"    交互数: 我们={ours['num_interactions']:,} vs 论文={paper['interactions']:,} {inter_match}")

all_results['Goodreads'] = gr_result

print(f"\n  ✅ Goodreads 处理完成!")

## 6. 保存处理结果供 Notebook 3 使用

将关键映射和过滤后的数据保存为中间文件，供下一个 Notebook 生成训练数据使用。

In [ ]:
import pickle

# 保存每个领域的 item_to_new_id 映射和序列数据
for category, result in all_results.items():
    save_data = {
        'item_to_new_id': result['item_to_new_id'],
        'item_titles': result['item_titles'],
        'data': result['data'],
        'train': result['train'],
        'val': result['val'],
        'test': result['test'],
        'stats': result['stats'],
    }
    pkl_path = INTERMEDIATE_DIR / f"{category}_processed.pkl"
    with open(pkl_path, 'wb') as f:
        pickle.dump(save_data, f)
    print(f"  ✅ 已保存: {pkl_path.name}")

print(f"\n所有处理结果已保存到 {INTERMEDIATE_DIR}")

## 7. 统计汇总与论文对照

In [ ]:
print("\n" + "="*100)
print("📊 5-Core 过滤后统计 — 与论文 Table 1/2 对照")
print("="*100)
print(f"{'数据集':<35} {'用户数':>10} {'物品数':>10} {'交互数':>12} {'论文物品':>10} {'论文交互':>12} {'匹配':>6}")
print("-"*100)

for category, result in all_results.items():
    stats = result['stats']
    paper = PAPER_STATS.get(category, {})
    
    if paper:
        items_ok = abs(stats['num_items'] - paper['items']) / paper['items'] < 0.05
        inter_ok = abs(stats['num_interactions'] - paper['interactions']) / paper['interactions'] < 0.05
        match_str = '✅' if (items_ok and inter_ok) else '⚠️'
        paper_items = f"{paper['items']:,}"
        paper_inter = f"{paper['interactions']:,}"
    else:
        match_str = 'N/A'
        paper_items = 'N/A'
        paper_inter = 'N/A'
    
    print(f"{category:<35} {stats['num_users']:>10,} {stats['num_items']:>10,} {stats['num_interactions']:>12,} {paper_items:>10} {paper_inter:>12} {match_str:>6}")

print("-"*100)
print("注: 由于预处理细节（如去重策略、时间戳精度等）的微小差异，数据可能与论文有 ±5% 的偏差")

## 8. 可视化

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")

# 序列长度分布
fig, axes = plt.subplots(3, 3, figsize=(16, 12))
axes = axes.flatten()

for idx, (category, result) in enumerate(all_results.items()):
    if idx >= 9:
        break
    ax = axes[idx]
    seq_lengths = [len(s) for s in result['data']]
    ax.hist(seq_lengths, bins=range(3, MAX_SEQ_LENGTH + 3), edgecolor='white', alpha=0.8, color='#3498db')
    ax.set_title(category.replace('_', ' '), fontsize=10)
    ax.set_xlabel('序列长度')
    ax.set_ylabel('用户数')
    ax.axvline(x=np.mean(seq_lengths), color='red', linestyle='--', alpha=0.8, label=f'均值={np.mean(seq_lengths):.1f}')
    ax.legend(fontsize=8)

plt.suptitle('5-Core 过滤后各数据集的用户序列长度分布', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# 与论文对比的柱状图
categories_with_paper = [c for c in all_results if PAPER_STATS.get(c, {})]

if categories_with_paper:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    x = range(len(categories_with_paper))
    width = 0.35
    
    # 物品数对比
    our_items = [all_results[c]['stats']['num_items'] for c in categories_with_paper]
    paper_items = [PAPER_STATS[c]['items'] for c in categories_with_paper]
    
    axes[0].bar([i - width/2 for i in x], our_items, width, label='我们', color='#3498db')
    axes[0].bar([i + width/2 for i in x], paper_items, width, label='论文', color='#e74c3c')
    axes[0].set_title('物品数对比')
    axes[0].set_xticks(list(x))
    axes[0].set_xticklabels([c.replace('_', '\n') for c in categories_with_paper], fontsize=7, rotation=45, ha='right')
    axes[0].legend()
    
    # 交互数对比
    our_inter = [all_results[c]['stats']['num_interactions'] for c in categories_with_paper]
    paper_inter = [PAPER_STATS[c]['interactions'] for c in categories_with_paper]
    
    axes[1].bar([i - width/2 for i in x], our_inter, width, label='我们', color='#3498db')
    axes[1].bar([i + width/2 for i in x], paper_inter, width, label='论文', color='#e74c3c')
    axes[1].set_title('交互数对比')
    axes[1].set_xticks(list(x))
    axes[1].set_xticklabels([c.replace('_', '\n') for c in categories_with_paper], fontsize=7, rotation=45, ha='right')
    axes[1].legend()
    
    plt.suptitle('我们的预处理 vs 论文报告的统计数据', fontsize=14)
    plt.tight_layout()
    plt.show()

## 9. 文件完整性检查

In [ ]:
print("📋 评估文件完整性检查")
print("="*80)

all_ok = True

# 检查 Amazon 领域
for category, output_path in AMAZON_OUTPUT_PATHS.items():
    dir_path = DATA_DIR / output_path
    expected_files = ['data.txt', 'train_data.txt', 'val_data.txt', 'test_data.txt', 'item_titles.json']
    
    print(f"\n  {category}:")
    for fname in expected_files:
        fp = dir_path / fname
        if fp.exists():
            size_kb = fp.stat().st_size / 1024
            print(f"    ✅ {fname:20s} ({size_kb:>8.1f} KB)")
        else:
            print(f"    ❌ {fname:20s} (缺失!)")
            all_ok = False

# 检查 Goodreads
gr_dir = DATA_DIR / 'Goodreads' / 'clean'
print(f"\n  Goodreads:")
for fname in ['data.txt', 'train_data.txt', 'val_data.txt', 'test_data.txt', 'item_titles.json']:
    fp = gr_dir / fname
    if fp.exists():
        size_kb = fp.stat().st_size / 1024
        print(f"    ✅ {fname:20s} ({size_kb:>8.1f} KB)")
    else:
        print(f"    ❌ {fname:20s} (缺失!)")
        all_ok = False

print(f"\n{'✅ 所有评估文件就绪!' if all_ok else '❌ 有文件缺失，请检查上述步骤'}")

In [ ]:
# 验证数据格式：读取一个 item_titles.json 并检查格式
print("🔍 数据格式验证 (以 Video_Games 为例):")
print("="*60)

sample_dir = DATA_DIR / 'Video_Games' / '5-core' / 'downstream'

# 检查 item_titles.json
with open(sample_dir / 'item_titles.json', 'r', encoding='utf-8') as f:
    titles = json.load(f)
print(f"\n📄 item_titles.json:")
print(f"  类型: {type(titles)}")
print(f"  物品数: {len(titles)}")
print(f"  最小ID: {min(int(k) for k in titles.keys())}  (应为1)")
print(f"  最大ID: {max(int(k) for k in titles.keys())}")
print(f"  是否包含'0': {'0' in titles}  (应为False)")
print(f"  前3个: {dict(list(titles.items())[:3])}")

# 检查 data.txt
with open(sample_dir / 'data.txt', 'r') as f:
    lines = f.readlines()
print(f"\n📄 data.txt:")
print(f"  行数: {len(lines)}")
print(f"  前3行:")
for line in lines[:3]:
    ids = list(map(int, line.strip().split()))
    print(f"    {ids} (长度={len(ids)})")

# 验证 leave-one-out 一致性
with open(sample_dir / 'train_data.txt', 'r') as f:
    train_lines = f.readlines()
with open(sample_dir / 'val_data.txt', 'r') as f:
    val_lines = f.readlines()
with open(sample_dir / 'test_data.txt', 'r') as f:
    test_lines = f.readlines()

print(f"\n📄 Leave-one-out 一致性验证:")
print(f"  data.txt 行数: {len(lines)}")
print(f"  train_data.txt 行数: {len(train_lines)}")
print(f"  val_data.txt 行数: {len(val_lines)}")
print(f"  test_data.txt 行数: {len(test_lines)}")
print(f"  行数是否一致: {'✅' if len(lines) == len(train_lines) == len(val_lines) == len(test_lines) else '❌'}")

# 验证第一个用户的划分
data_seq = list(map(int, lines[0].strip().split()))
train_seq = list(map(int, train_lines[0].strip().split()))
val_seq = list(map(int, val_lines[0].strip().split()))
test_seq = list(map(int, test_lines[0].strip().split()))

print(f"\n  第1个用户的序列:")
print(f"    data:  {data_seq}")
print(f"    test:  {test_seq}  (= data，完整序列)")
print(f"    val:   {val_seq}  (= data[:-1])")
print(f"    train: {train_seq}  (= data[:-2])")
print(f"    test == data: {'✅' if test_seq == data_seq else '❌'}")
print(f"    val == data[:-1]: {'✅' if val_seq == data_seq[:-1] else '❌'}")
print(f"    train == data[:-2]: {'✅' if train_seq == data_seq[:-2] else '❌'}")

## 📝 小结

本 Notebook 完成了以下工作：

1. ✅ 对 8 个 Amazon 领域 + Goodreads 执行迭代式 5-core 过滤
2. ✅ 物品 ID 从 1 开始重编号（0 保留作为 padding）
3. ✅ 按时间排序构建用户交互序列（最大长度 10）
4. ✅ 采用 leave-one-out 策略划分 train/val/test
5. ✅ 生成全部评估文件（data.txt, train_data.txt, val_data.txt, test_data.txt, item_titles.json）
6. ✅ 与论文统计数据对照验证

### 下一步
→ 打开 `03_generate_training_data.ipynb`，生成 CSFT/MNTP/SimCSE 训练数据。